In [14]:

import os
import shutil


from google.colab import userdata
USER = "Koubra-gaby"
#TOKEN = userdata.get('GH_TOKEN')
REPO_NAME = "Chadian_arabic-update"

if os.path.exists(REPO_NAME):
    print(f"Le dossier '{REPO_NAME}' existe déjà. Suppression en cours...")
    shutil.rmtree(REPO_NAME)
# Cloner le dépôt privé
!git clone https://{USER}:@github.com/{USER}/{REPO_NAME}.git


Le dossier 'Chadian_arabic-update' existe déjà. Suppression en cours...
Cloning into 'Chadian_arabic-update'...
remote: Enumerating objects: 295, done.
remote: Counting objects: 100% (295/295), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 295 (delta 133), reused 221 (delta 76), pack-reused 0 (from 0)
Receiving objects: 100% (295/295), 15.66 MiB | 13.46 MiB/s, done.
Resolving deltas: 100% (133/133), done.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token = userdata.get('update_shu'))

In [ ]:

import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def backtranslation_dataset(
    fichier_txt,
    nom_modele="facebook/nllb-200-distilled-600M",
    langue_source="fra_Latn",
    langue_cible="arb_Latn",
    test_size=0.10,
    val_size=0.10,
    batch_size=8,
    seed=42,
):
    # 1. Charger le corpus français
    with open(fichier_txt, "r", encoding="utf-8-sig") as f:
        phrases_fr = list(dict.fromkeys(
            ligne.strip() for ligne in f if ligne.strip()
        ))

    if len(phrases_fr) < 3:
        raise ValueError("Le fichier doit contenir au moins 3 phrases.")

    if test_size <= 0 or val_size <= 0 or test_size + val_size >= 1:
        raise ValueError("Les proportions doivent être positives et leur somme < 1.")

    # 2. Charger le modèle NLLB
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(
        nom_modele,
        src_lang=langue_source,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(nom_modele).to(device)
    model.eval()

    target_id = tokenizer.convert_tokens_to_ids(langue_cible)

    if target_id == tokenizer.unk_token_id:
        raise ValueError(f"Code de langue cible inconnu : {langue_cible}")

    # 3. Générer les traductions vers la langue cible
    phrases_arb = []

    with torch.inference_mode():
        for i in range(0, len(phrases_fr), batch_size):
            batch_fr = phrases_fr[i:i + batch_size]

            inputs = tokenizer(
                batch_fr,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            ).to(device)

            outputs = model.generate(
                **inputs,
                forced_bos_token_id=target_id,
                max_new_tokens=256,
                num_beams=4,
            )

            phrases_arb.extend(
                tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True,
                )
            )

            print(f"{min(i + batch_size, len(phrases_fr))}/{len(phrases_fr)}")

    # 4. Construire exactement la structure translation
    dataset_dict = {
        "translation": [
            {"arb": arb, "fr": fr}
            for arb, fr in zip(phrases_arb, phrases_fr)
        ]
    }

    dataset = Dataset.from_dict(dataset_dict)

    # 5. Créer train, val et test
    split_test = dataset.train_test_split(
        test_size=test_size,
        seed=seed,
    )

    train_val = split_test["train"].train_test_split(
        test_size=val_size / (1 - test_size),
        seed=seed,
    )

    resultat = DatasetDict({
        "train": train_val["train"],
        "val": train_val["test"],
        "test": split_test["test"],
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(resultat)
    print("\nExemple :", resultat["train"][0])

    return resultat


### Utilisation

In [8]:
dataset_dict_not_bible = backtranslation_dataset(
    fichier_txt="/content/Chadian_arabic-update/Data/synthetics_dataset/molingal_data_french.txt",
    nom_modele="Koubra-Gaby/nllb-200-distilled-600M",
    langue_source="fra_Latn",
    langue_cible="arb_Latn",
)

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


8/56451


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


16/56451


...

56432/56451


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


56440/56451


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


56448/56451
56451/56451
DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 45160
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 5645
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 5646
    })
})

Exemple : {'translation': {'arb': 'sana al-sakhiir al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye,al-jigindiiye', 'fr': 'En 2023, le festival Rio Loco proposait lors de sa soirée d’ouverture un hommage au batteur nigérian Tony Allen (1940-2020), qui avait rassemblé 16 000 spectateurs sur la prairie des Filtres, en bord de Garonne.'}}


In [9]:
dataset_dict_not_bible

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 45160
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 5645
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 5646
    })
})

### Concater avec le datadict existant

In [10]:
from datasets import load_from_disk
train_test_val_dataset_bible = load_from_disk("/content/Chadian_arabic-update/Data/train_test_split")

In [11]:
from datasets import concatenate_datasets, DatasetDict

train_test_val_dataset = DatasetDict({
    split: concatenate_datasets([
        train_test_val_dataset_bible[split],
        dataset_dict_not_bible[split],
    ]).shuffle(seed=42)
    for split in ["train", "val", "test"]
})

In [ ]:
#dataset_combine.save_to_disk("train_test_val_dataset_bible")

### Entrainement

In [15]:
%pip install evaluate
%pip install sacrebleu
! pip install huggingface_hub
%pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=2d2b99836170ec95aeb141279e216375bf3f66c9ee1185e82a7e2cd71f8dde39
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [16]:
import os
import sys
module_path = os.path.abspath(os.path.join('/content/Chadian_arabic-update'))
if module_path not in sys.path:
    sys.path.append(module_path)


In [17]:
import evaluate
from src.models import training, plot_loss_curves
from datasets import load_from_disk
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback

In [18]:
nllb = "facebook/nllb-200-distilled-600M"

In [1]:
trainer_nllb, nllb_evaluation = training(nllb, train_test_val_dataset)

nllb_evaluation

NameError: name 'training' is not defined

In [ ]:
plot_loss_curves(trainer_nllb, nllb)

In [ ]:
trainer_nllb.push_to_hub()
